# CamemBERT Fine-Tuning for Classification

---
## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Fixed: AdamW is now in torch.optim
from transformers import CamembertTokenizer, CamembertForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Ti
Memory: 8.22 GB


---
## 2. Load Data from features.pkl

Loading from the same file used for embeddings approach (contains original text + metadata).

In [2]:
print("Loading data from features.pkl...")

import pickle
with open('features.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = pd.read_pickle('original_data/data_train.pkl')
X_kaggle = pd.read_pickle('original_data/data_kaggle.pkl')
y_train = data['y_train']
kaggle_ids = data['kaggle_ids']

print(f"✓ Data loaded")
print(f"  Train samples: {len(X_train)}")
print(f"  Kaggle samples: {len(X_kaggle)}")
print(f"\nClass distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count/len(y_train)*100:.2f}%)")

Loading data from features.pkl...
✓ Data loaded
  Train samples: 154914
  Kaggle samples: 103380

Class distribution:
  Class 0: 82674 (53.37%)
  Class 1: 72240 (46.63%)


---
## 3. Configuration

In [ ]:
# ============ TRAINING CONFIGURATION ============

# Model
MODEL_NAME = 'camembert-base'
MAX_LENGTH = 280  # Maximum sequence length

# Training
BATCH_SIZE = 16   # Reduce if you get OOM errors
NUM_EPOCHS = 3    # Fine-tuning usually needs fewer epochs
LEARNING_RATE = 2e-5  # Lower LR for fine-tuning
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

# Text feature to use
TEXT_COLUMN = 'full_text'  # Main text column for classification

# ================================================

print("Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Text column: {TEXT_COLUMN}")

---
## 4. Prepare Text Data

In [ ]:
# Fill missing values
print("Preparing text data...")

X_train[TEXT_COLUMN] = X_train[TEXT_COLUMN].fillna('')
X_kaggle[TEXT_COLUMN] = X_kaggle[TEXT_COLUMN].fillna('')

# Extract text and labels
train_texts = X_train[TEXT_COLUMN].tolist()
train_labels = y_train  # Already numpy array from features.pkl
kaggle_texts = X_kaggle[TEXT_COLUMN].tolist()
# kaggle_ids already loaded from features.pkl

print(f"✓ Text data prepared")
print(f"  Train texts: {len(train_texts)}")
print(f"  Kaggle texts: {len(kaggle_texts)}")
print(f"\nExample text (first 200 chars):")
print(f"  {train_texts[0][:200]}...")

---
## 5. Create Dataset and DataLoader

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = CamembertTokenizer.from_pretrained(MODEL_NAME)
print("✓ Tokenizer loaded")

# Custom Dataset
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length, labels=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return item

print("\nCreating datasets...")
train_dataset = TextDataset(train_texts, tokenizer, MAX_LENGTH, train_labels)
kaggle_dataset = TextDataset(kaggle_texts, tokenizer, MAX_LENGTH)

print(f"✓ Datasets created")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

kaggle_loader = DataLoader(
    kaggle_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✓ DataLoaders ready")
print(f"  Train batches: {len(train_loader)}")
print(f"  Kaggle batches: {len(kaggle_loader)}")

---
## 6. Initialize Model

In [ ]:
print("Initializing CamemBERT for sequence classification...")

# Load pre-trained model with classification head
model = CamembertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # Binary classification
)

model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Model initialized")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e9:.2f} GB")

---
## 7. Setup Optimizer and Scheduler

In [ ]:
# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print("✓ Optimizer and scheduler configured")
print(f"  Total training steps: {total_steps}")
print(f"  Warmup steps: {WARMUP_STEPS}")

---
## 8. Training Loop

In [ ]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(data_loader, desc="Training")
    
    for batch in progress_bar:
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()
        
        # Calculate accuracy
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{correct/total:.4f}'
        })
    
    return total_loss / len(data_loader), correct / total

print("✓ Training function defined")

In [ ]:
# Train the model
print(f"\nStarting fine-tuning for {NUM_EPOCHS} epochs...")
print("="*70)

best_acc = 0
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 70)
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    
    print(f"\nEpoch {epoch + 1} Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Train Accuracy: {train_acc*100:.2f}%")
    print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.2e}")
    
    # Save best model
    if train_acc > best_acc:
        best_acc = train_acc
        torch.save(model.state_dict(), 'camembert_finetuned_best.pth')
        print(f"  ✓ New best accuracy! Model saved.")

print("\n" + "="*70)
print(f"✓ Training complete!")
print(f"  Best training accuracy: {best_acc*100:.2f}%")
print("="*70)

---
## 9. Generate Predictions

In [ ]:
def predict(model, data_loader, device):
    """Generate predictions"""
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
    
    return predictions

print("✓ Prediction function defined")

In [ ]:
# Load best model and predict
print("\nLoading best model for predictions...")
model.load_state_dict(torch.load('camembert_finetuned_best.pth'))

print("Generating predictions on Kaggle test set...")
predictions = predict(model, kaggle_loader, device)

print(f"\n✓ Predictions generated")
print(f"\nPrediction distribution:")
unique, counts = np.unique(predictions, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count/len(predictions)*100:.2f}%)")

---
## 10. Save Predictions

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'ID': kaggle_ids,
    'Prediction': predictions
})

submission.to_csv('camembert_finetuned_predictions.csv', index=False)

print("\n✓ Predictions saved to 'camembert_finetuned_predictions.csv'")
print(f"\nSubmission file preview:")
print(submission.head(10))
print(f"\nTotal predictions: {len(submission)}")

---
## 11. Save Full Model (Optional)

In [ ]:
# Save the complete model (can be loaded later)
model.save_pretrained('camembert_finetuned_model')
tokenizer.save_pretrained('camembert_finetuned_model')

print("✓ Full model and tokenizer saved to 'camembert_finetuned_model/'")
print("\nTo load later:")
print("  model = CamembertForSequenceClassification.from_pretrained('camembert_finetuned_model')")
print("  tokenizer = CamembertTokenizer.from_pretrained('camembert_finetuned_model')")